# Notebook 6: Simulation-Based Regression Analysis

This notebook runs regression models comparing actual versus simulated (null model) evacuation destinations, quantifying the role of behavioral preferences beyond what is explained by the gravity model.

**Paper reference:** Figure 3b — Regression on difference between actual and simulated destinations; Supplementary Tables S15–S20.

**Inputs:** `evacuees_newSCI_simulation.csv` (output of Notebook 5b)

**Outputs:** Delta similarity/connectedness regression results, Stargazer tables


In [ ]:
import geopandas as gpd
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import statsmodels.api as sm
import seaborn as sns

In [ ]:
import itertools
from shapely.geometry import Point
from geopy.distance import geodesic
from sklearn.preprocessing import StandardScaler

In [ ]:
evacuees = pd.read_csv("evacuees_newSCI_simulation.csv")
evacuees.head()

In [ ]:
evacuees.columns

In [ ]:
simulated_destination = pd.read_csv("simulated_location_revised.csv.gz")


In [ ]:
simulated_destination = simulated_destination.rename(columns = {'scaled_sci' : 'Familiarity'})
simulated_destination.columns

In [ ]:
evacuees['pre_crisis_home_cbg'] = evacuees['pre_crisis_home_cbg'].astype(str)
simulated_destination['pre_crisis_home_cbg'] = simulated_destination['pre_crisis_home_cbg'].astype(str)

In [ ]:
simulated_destination_for_reg = pd.merge(
    simulated_destination, 
    evacuees, 
    left_on=['pre_crisis_home_cbg', 'crisis_home_cbg'],
    right_on=['pre_crisis_home_cbg', 'crisis_home_cbg'], 
    how='left' 
    )
simulated_destination_for_reg.head()

# Regressions_simulated

In [ ]:
simulated_destination_for_reg.columns

# difference = actual - simulated

In [ ]:
df = simulated_destination_for_reg.dropna()
df['log_Familiarity'] = df['Familiarity'].apply(np.log1p)
df['log_scaled_sci'] = df['scaled_sci'].apply(np.log1p)
df['difference_Similarity'] =  df['Similarity'] - df['homophily'] 
df['difference_Familiarity'] =  df['log_scaled_sci'] - df['log_Familiarity'] 
df['log_distance_OD_y'] = df['distance_OD_y'].apply(np.log1p)
df['log_Or_black_population'] = df['Or_black_population'].apply(np.log1p)
df['log_Or_white_population'] = df['Or_white_population'].apply(np.log1p)

In [ ]:
df.head()

In [ ]:
Y = df['difference_Similarity'] 
df['log_distance_OD_y'] = df['distance_OD_y'].apply(np.log1p)

independent_vars = [
                        # 'Or_population_density', 
                      #'log_Or_unemployment_rate', 
                     'log_Or_white_population',
#                     'log_black_population',
#                      #'log_Or_asian_population',
#                      'log_Or_all_others',
                    # 'Or_median_household_income',
                      #'Or_education_atleast_one_degree', 
                      'log_distance_OD_y',
                      #'distance_OD_y',
                    #  'avg_entropy',
                    # 'radius_of_gyration'
                    ]
# Standardize the features
scaler = StandardScaler()
df[independent_vars] = scaler.fit_transform(df[independent_vars])

# Prepare the independent variables DataFrame
X = df[independent_vars]

# Add a constant term (intercept) to the independent variables
X_with_const = sm.add_constant(X)

# Perform the regression
model_homo= sm.OLS(Y, X_with_const).fit()

# Print the regression summary
print("Regression summary for Similarity:\n")
print(model_homo.summary())


In [ ]:
Y = df['difference_Familiarity'] 

independent_vars = [
    # 'Or_population_density', 
                     # 'log_Or_unemployment_rate', 
                  'log_Or_white_population',
                    #'log_Or_black_population',
    #                  'log_Or_all_others',
    #                  #'log_Or_asian_population',  
                      #'Or_median_household_income',
                    # 'Or_education_atleast_one_degree', 
    #                  'dist_epicentre',
                      'log_distance_OD_y',
    #                  'avg_entropy',
    #                 'radius_of_gyration'
                    ]
# Standardize the features
scaler = StandardScaler()
df[independent_vars] = scaler.fit_transform(df[independent_vars])

# Prepare the independent variables DataFrame
X = df[independent_vars]

# Add a constant term (intercept) to the independent variables
X_with_const = sm.add_constant(X)

# Perform the regression
model_SCI= sm.OLS(Y, X_with_const).fit()

# Print the regression summary
print("Regression summary for Familiarity:\n")
print(model_SCI.summary())


In [ ]:
# Define the models with dependent and independent variables
models = [
    {"y_var": "difference_Familiarity", "x_vars": ["log_distance_OD_y",  "log_Or_white_population"]},
    {"y_var": "difference_Familiarity", "x_vars": ["log_distance_OD_y",  "log_Or_black_population"]},
    {"y_var": "difference_Familiarity", "x_vars": ["log_distance_OD_y",  "Or_median_household_income"]},
    {"y_var": "difference_Similarity", "x_vars": ["log_distance_OD_y",  "log_Or_white_population"]},
    {"y_var": "difference_Similarity", "x_vars": ["log_distance_OD_y",  "log_Or_black_population"]},
    {"y_var": "difference_Similarity", "x_vars": ["log_distance_OD_y",  "Or_median_household_income"]}
]

# Initialize an empty list to store results
results_list = []

# Loop through each model configuration
for model_info in models:
    y_var = model_info["y_var"]
    x_vars = model_info["x_vars"]
    
    # Define Y and X using the cleaned DataFrame
    Y = df[[y_var]]  # Convert Y to a DataFrame for scaling
    X = df[x_vars]
    
    # Standardize X variables
    x_scaler = StandardScaler()
    X_scaled = x_scaler.fit_transform(X)
    X_scaled = sm.add_constant(X_scaled)  # Add constant (intercept) term
    
    # Fit the regression model
    model = sm.OLS(Y, X_scaled).fit()
    
    # Extract confidence intervals
    conf_int = model.conf_int(alpha=0.05)  # 95% confidence intervals

    # Extract model summary statistics
    r_squared = model.rsquared
    for i, (var_name, coef, std_err, t_value, p_value) in enumerate(
        zip(x_vars, model.params[1:], model.bse[1:], model.tvalues[1:], model.pvalues[1:])
    ):
        lower_ci, upper_ci = conf_int.iloc[i + 1]  # Adjust index to skip the intercept
        results_list.append({
            "y_variable": y_var,
            "x_variable": var_name,
            "R_squared": r_squared,
            "Coefficient": round(coef, 2),
            "Std_Error": round(std_err, 4),
            "t_value": round(t_value, 2),
            "p_value": round(p_value, 2),
            "CI_Lower": round(lower_ci, 2),  # Lower bound of confidence interval
            "CI_Upper": round(upper_ci, 2)   # Upper bound of confidence interval
        })

# Convert results to a DataFrame
results_df = pd.DataFrame(results_list)

# Display the results
results_df

# Convert results to a DataFrame
results_df = pd.DataFrame(results_list)

# Display the results
results_df

In [ ]:
import matplotlib.pyplot as plt

def plot_selected_coefficients(df, y_vars, selected_x_vars, significance_level=0.05):
    # Define colors for each y variable for significant variables
    color_mapping = {
        'difference_Familiarity': 'orange',  # Chocolate brown color for significant coefficients in log_scaled_sci models
        'difference_Similarity': 'steelblue'  # Steel blue color for significant coefficients in homophily models
    }
    
    # Grey color for insignificant variables
    insignificance_color = 'grey'
    
    # Loop over each y variable to create a separate plot
    for y_var in y_vars:
        # Filter the results DataFrame to get only the rows for this y variable and selected x variables
        y_df = df[(df['y_variable'] == y_var) & (df['x_variable'].isin(selected_x_vars))]

        # Check the contents of y_df for debugging
        print(f"\nData for {y_var}:")
        print(y_df[['x_variable', 'Coefficient', 'CI_Lower', 'CI_Upper', 'p_value']])

        # Ensure the x_variables are ordered as specified in `selected_x_vars`
        y_df['x_variable'] = pd.Categorical(
            y_df['x_variable'], categories=selected_x_vars, ordered=True
        )
        y_df = y_df.sort_values('x_variable')

        # Set the color for significant coefficients based on y_variable
        significant_color = color_mapping.get(y_var, 'green')  # Default to green if y_var is not found

        # Plot
        fig, ax = plt.subplots(figsize=(5, 2))
        ax.invert_yaxis()
        
        # Plot each coefficient with error bars, using confidence intervals
        for i, row in y_df.iterrows():
            # Choose color based on significance
            plot_color = significant_color if row['p_value'] < significance_level else insignificance_color
            
            # Calculate error bar values using confidence intervals
            ci_lower = row['Coefficient'] - row['CI_Lower']
            ci_upper = row['CI_Upper'] - row['Coefficient']
            
            # Plot error bar for each coefficient
            ax.errorbar(row['Coefficient'], row['x_variable'], 
                        xerr=[[ci_lower], [ci_upper]], fmt='v', color=plot_color,
                        ecolor='black', elinewidth=0.8, capsize=4, markersize=8, linestyle='None',
                        markerfacecolor=plot_color, markeredgewidth=1.5, alpha=0.8)

        # Customize plot appearance
        ax.axvline(0, color='grey', linewidth=0.8, linestyle='--')
        # ax.set_xlabel('Beta Coefficients')
        # ax.set_ylabel('Variables')
        # ax.set_title(f'{y_var}')

        # Remove all spines (boundaries) to leave only tick marks
        ax.spines['top'].set_visible(False)
        ax.spines['right'].set_visible(False)
        ax.spines['left'].set_visible(False)
        ax.spines['bottom'].set_visible(False)

        # Show only tick marks without axis lines
        ax.xaxis.set_ticks_position('bottom')
        ax.yaxis.set_ticks_position('left')

        plt.tight_layout()
        # Save the plot with an appropriate filename
        plot_filename = f"{y_var}_difference_coeff.png"
        plt.savefig(plot_filename, dpi=300, bbox_inches='tight')
        print(f"Plot saved as {plot_filename}")
        plt.show()

# Define the y variables and selected x variables
y_variables = ['difference_Familiarity', 'difference_Similarity']  # Corrected to match the actual y_variable names
selected_x_variables = ['log_Or_white_population', 'log_Or_black_population', 'Or_median_household_income']

# Call the function
plot_selected_coefficients(results_df, y_variables, selected_x_variables)

# Stargazer

In [ ]:
Y = df['difference_Similarity'] 
df['log_distance_OD_y'] = df['distance_OD_y'].apply(np.log1p)

independent_vars = ['log_Or_white_population',
                      'log_distance_OD_y'
                     ]

scaler = StandardScaler()
df[independent_vars] = scaler.fit_transform(df[independent_vars])
X = df[independent_vars]
X_with_const = sm.add_constant(X)
model_sim_white= sm.OLS(Y, X_with_const).fit()


In [ ]:
params = model_sim_white.params.round(4)
pvalues = model_sim_white.pvalues.round(4)
conf_int = model_sim_white.conf_int().round(4)

variable_names = {
    'log_Or_white_population': 'White percent',
    'log_distance_OD': 'Evacuation Distance'
}

latex_output = r"""
\begin{table}[!htbp] \centering
  \caption{Regression Results: $\delta_$ Similarity = Evacuation Distance + White percent}
\begin{tabular}{@{\extracolsep{5pt}}lc}
\hline \hline
& \multicolumn{1}{c}{\textit{Dependent variable: \delta_Similarity}} \\
\cline{2-2}
\\[-1.8ex] & \textbf{Model 1} \\
\hline \\[-1.8ex]
"""

for var, coef in params.items():
    # Get the renamed variable, or default to the original name
    var_name = variable_names.get(var, var).title()
    pval = pvalues[var]
    lower_ci, upper_ci = conf_int.loc[var]
    
    latex_output += f"{var_name} & {coef} ($p={pval}$) \\\\\n"
    latex_output += f"& [{lower_ci}, {upper_ci}] \\\\\n"

latex_output += r"""
\hline \\[-1.8ex]
Observations & %d \\
$R^2$ & %.3f \\
Adjusted $R^2$ & %.3f \\
\hline
\hline \\[-1.8ex]
\textit{Note:} & \multicolumn{1}{r}{$^{*}$p$<$0.1; $^{**}$p$<$0.05; $^{***}$p$<$0.01} \\
\end{tabular}
\end{table}
""" % (model_sim_white.nobs, model_sim_white.rsquared, model_sim_white.rsquared_adj)

print(latex_output)

In [ ]:
Y = df['difference_Similarity'] 
df['log_distance_OD_y'] = df['distance_OD_y'].apply(np.log1p)

independent_vars = ['log_Or_black_population',
                      'log_distance_OD_y'
                     ]

scaler = StandardScaler()
df[independent_vars] = scaler.fit_transform(df[independent_vars])
X = df[independent_vars]
X_with_const = sm.add_constant(X)
model_sim_black= sm.OLS(Y, X_with_const).fit()


In [ ]:
params = model_sim_black.params.round(4)
pvalues = model_sim_black.pvalues.round(4)
conf_int = model_sim_black.conf_int().round(4)

variable_names = {
    'log_Or_black_population': 'Black percent',
    'log_distance_OD': 'Evacuation Distance'
}

latex_output = r"""
\begin{table}[!htbp] \centering
  \caption{Regression Results: $\delta_$ Similarity = Evacuation Distance + Black percent}
\begin{tabular}{@{\extracolsep{5pt}}lc}
\hline \hline
& \multicolumn{1}{c}{\textit{Dependent variable: \delta_Similarity}} \\
\cline{2-2}
\\[-1.8ex] & \textbf{Model 1} \\
\hline \\[-1.8ex]
"""

for var, coef in params.items():
    # Get the renamed variable, or default to the original name
    var_name = variable_names.get(var, var).title()
    pval = pvalues[var]
    lower_ci, upper_ci = conf_int.loc[var]
    
    latex_output += f"{var_name} & {coef} ($p={pval}$) \\\\\n"
    latex_output += f"& [{lower_ci}, {upper_ci}] \\\\\n"

latex_output += r"""
\hline \\[-1.8ex]
Observations & %d \\
$R^2$ & %.3f \\
Adjusted $R^2$ & %.3f \\
\hline
\hline \\[-1.8ex]
\textit{Note:} & \multicolumn{1}{r}{$^{*}$p$<$0.1; $^{**}$p$<$0.05; $^{***}$p$<$0.01} \\
\end{tabular}
\end{table}
""" % (model_sim_black.nobs, model_sim_black.rsquared, model_sim_black.rsquared_adj)

print(latex_output)

In [ ]:
Y = df['difference_Similarity'] 
df['log_distance_OD_y'] = df['distance_OD_y'].apply(np.log1p)

independent_vars = ['Or_median_household_income',
                      'log_distance_OD_y'
                     ]

scaler = StandardScaler()
df[independent_vars] = scaler.fit_transform(df[independent_vars])
X = df[independent_vars]
X_with_const = sm.add_constant(X)
model_sim_inc= sm.OLS(Y, X_with_const).fit()


In [ ]:
params = model_sim_inc.params.round(4)
pvalues = model_sim_inc.pvalues.round(4)
conf_int = model_sim_inc.conf_int().round(4)

variable_names = {
    'Or_median_household_income': 'Median income',
    'log_distance_OD': 'Evacuation Distance'
}

latex_output = r"""
\begin{table}[!htbp] \centering
  \caption{Regression Results: $\delta_$ Similarity = Evacuation Distance + Median income}
\begin{tabular}{@{\extracolsep{5pt}}lc}
\hline \hline
& \multicolumn{1}{c}{\textit{Dependent variable: \delta_Similarity}} \\
\cline{2-2}
\\[-1.8ex] & \textbf{Model 1} \\
\hline \\[-1.8ex]
"""

for var, coef in params.items():
    # Get the renamed variable, or default to the original name
    var_name = variable_names.get(var, var).title()
    pval = pvalues[var]
    lower_ci, upper_ci = conf_int.loc[var]
    
    latex_output += f"{var_name} & {coef} ($p={pval}$) \\\\\n"
    latex_output += f"& [{lower_ci}, {upper_ci}] \\\\\n"

latex_output += r"""
\hline \\[-1.8ex]
Observations & %d \\
$R^2$ & %.3f \\
Adjusted $R^2$ & %.3f \\
\hline
\hline \\[-1.8ex]
\textit{Note:} & \multicolumn{1}{r}{$^{*}$p$<$0.1; $^{**}$p$<$0.05; $^{***}$p$<$0.01} \\
\end{tabular}
\end{table}
""" % (model_sim_inc.nobs, model_sim_inc.rsquared, model_sim_inc.rsquared_adj)

print(latex_output)

In [ ]:
Y = df['difference_Familiarity'] 
df['log_distance_OD_y'] = df['distance_OD_y'].apply(np.log1p)

independent_vars = ['log_Or_white_population',
                      'log_distance_OD_y'
                     ]

scaler = StandardScaler()
df[independent_vars] = scaler.fit_transform(df[independent_vars])
X = df[independent_vars]
X_with_const = sm.add_constant(X)
model_fam_white= sm.OLS(Y, X_with_const).fit()


In [ ]:
params = model_fam_white.params.round(4)
pvalues = model_fam_white.pvalues.round(4)
conf_int = model_fam_white.conf_int().round(4)

variable_names = {
    'log_Or_white_population': 'White percent',
    'log_distance_OD': 'Evacuation Distance'
}

latex_output = r"""
\begin{table}[!htbp] \centering
  \caption{Regression Results: $\delta_$ Connectedness = Evacuation Distance + White percent}
\begin{tabular}{@{\extracolsep{5pt}}lc}
\hline \hline
& \multicolumn{1}{c}{\textit{Dependent variable: $\delta_$ Connectedness}} \\
\cline{2-2}
\\[-1.8ex] & \textbf{Model 1} \\
\hline \\[-1.8ex]
"""

for var, coef in params.items():
    # Get the renamed variable, or default to the original name
    var_name = variable_names.get(var, var).title()
    pval = pvalues[var]
    lower_ci, upper_ci = conf_int.loc[var]
    
    latex_output += f"{var_name} & {coef} ($p={pval}$) \\\\\n"
    latex_output += f"& [{lower_ci}, {upper_ci}] \\\\\n"

latex_output += r"""
\hline \\[-1.8ex]
Observations & %d \\
$R^2$ & %.3f \\
Adjusted $R^2$ & %.3f \\
\hline
\hline \\[-1.8ex]
\textit{Note:} & \multicolumn{1}{r}{$^{*}$p$<$0.1; $^{**}$p$<$0.05; $^{***}$p$<$0.01} \\
\end{tabular}
\end{table}
""" % (model_fam_white.nobs, model_fam_white.rsquared, model_fam_white.rsquared_adj)

print(latex_output)

In [ ]:
Y = df['difference_Familiarity'] 
df['log_distance_OD_y'] = df['distance_OD_y'].apply(np.log1p)

independent_vars = ['log_Or_black_population',
                      'log_distance_OD_y'
                     ]

scaler = StandardScaler()
df[independent_vars] = scaler.fit_transform(df[independent_vars])
X = df[independent_vars]
X_with_const = sm.add_constant(X)
model_fam_black= sm.OLS(Y, X_with_const).fit()


In [ ]:
params = model_fam_black.params.round(4)
pvalues = model_fam_black.pvalues.round(4)
conf_int = model_fam_black.conf_int().round(4)

variable_names = {
    'log_Or_black_population': 'Black percent',
    'log_distance_OD': 'Evacuation Distance'
}

latex_output = r"""
\begin{table}[!htbp] \centering
  \caption{Regression Results: $\delta_$ Connectedness = Evacuation Distance + Black percent}
\begin{tabular}{@{\extracolsep{5pt}}lc}
\hline \hline
& \multicolumn{1}{c}{\textit{Dependent variable: $\delta_$ Connectedness}} \\
\cline{2-2}
\\[-1.8ex] & \textbf{Model 1} \\
\hline \\[-1.8ex]
"""

for var, coef in params.items():
    # Get the renamed variable, or default to the original name
    var_name = variable_names.get(var, var).title()
    pval = pvalues[var]
    lower_ci, upper_ci = conf_int.loc[var]
    
    latex_output += f"{var_name} & {coef} ($p={pval}$) \\\\\n"
    latex_output += f"& [{lower_ci}, {upper_ci}] \\\\\n"

latex_output += r"""
\hline \\[-1.8ex]
Observations & %d \\
$R^2$ & %.3f \\
Adjusted $R^2$ & %.3f \\
\hline
\hline \\[-1.8ex]
\textit{Note:} & \multicolumn{1}{r}{$^{*}$p$<$0.1; $^{**}$p$<$0.05; $^{***}$p$<$0.01} \\
\end{tabular}
\end{table}
""" % (model_fam_black.nobs, model_fam_black.rsquared, model_fam_black.rsquared_adj)

print(latex_output)

In [ ]:
Y = df['difference_Familiarity'] 
df['log_distance_OD_y'] = df['distance_OD_y'].apply(np.log1p)

independent_vars = ['Or_median_household_income',
                      'log_distance_OD_y'
                     ]

scaler = StandardScaler()
df[independent_vars] = scaler.fit_transform(df[independent_vars])
X = df[independent_vars]
X_with_const = sm.add_constant(X)
model_fam_inc= sm.OLS(Y, X_with_const).fit()


In [ ]:
params = model_fam_inc.params.round(4)
pvalues = model_fam_inc.pvalues.round(4)
conf_int = model_fam_inc.conf_int().round(4)

variable_names = {
    'Or_median_household_income': 'Median income',
    'log_distance_OD': 'Evacuation Distance'
}

latex_output = r"""
\begin{table}[!htbp] \centering
  \caption{Regression Results: $\delta_$ Connectedness = Evacuation Distance + Median income}
\begin{tabular}{@{\extracolsep{5pt}}lc}
\hline \hline
& \multicolumn{1}{c}{\textit{Dependent variable: $\delta_$ Connectedness}} \\
\cline{2-2}
\\[-1.8ex] & \textbf{Model 1} \\
\hline \\[-1.8ex]
"""

for var, coef in params.items():
    # Get the renamed variable, or default to the original name
    var_name = variable_names.get(var, var).title()
    pval = pvalues[var]
    lower_ci, upper_ci = conf_int.loc[var]
    
    latex_output += f"{var_name} & {coef} ($p={pval}$) \\\\\n"
    latex_output += f"& [{lower_ci}, {upper_ci}] \\\\\n"

latex_output += r"""
\hline \\[-1.8ex]
Observations & %d \\
$R^2$ & %.3f \\
Adjusted $R^2$ & %.3f \\
\hline
\hline \\[-1.8ex]
\textit{Note:} & \multicolumn{1}{r}{$^{*}$p$<$0.1; $^{**}$p$<$0.05; $^{***}$p$<$0.01} \\
\end{tabular}
\end{table}
""" % (model_fam_inc.nobs, model_fam_inc.rsquared, model_fam_inc.rsquared_adj)

print(latex_output)

In [ ]:
from stargazer.stargazer import Stargazer, LineLocation

In [ ]:
stargazer = Stargazer([model_sim_inc])

stargazer.title("Regression Results for $\Delta$ Similarity ~ Median Income ")
stargazer.custom_columns(["Model 1"], [1])
stargazer.show_degrees_of_freedom(False)                 
stargazer

In [ ]:
latex_output = stargazer.render_latex()
print(latex_output)

In [ ]:
thresholds = [240, 480, 720, 960, 1200, 1440, 1680, 1920]
evacuees = [21453, 20062, 17255, 10575, 5802, 248, 207, 142]

plt.figure(figsize=(8, 4.5))
plt.plot(thresholds, evacuees, marker='o', label='Evacuees', color='blue')
plt.plot(thresholds, non_evacuees, marker='o', label='Non-Evacuees', color='orange')

plt.xticks(thresholds, labels=[str(t) for t in thresholds])

plt.xlabel('Thresholds (meters)', fontsize=12)
plt.ylabel('Number of People', fontsize=12)
plt.title('Tipping Point Analysis: Evacuees vs. Non-Evacuees', fontsize=14)
plt.legend()
plt.grid(alpha=0.5)
plt.tight_layout()

plt.savefig('tipping_point.png', dpi=300, bbox_inches='tight')
plt.show()

# Superseded

In [ ]:
df = simulated_destination_for_reg.dropna()
df['log_scaled_sci'] = df['scaled_sci'].apply(np.log1p) 
Y_SCI = df['log_scaled_sci']
Y_homo = df['Similarity']

df['log_radius_of_gyration'] = df['radius_of_gyration'].apply(np.log1p) 
df['log_distance'] = df['distance'].apply(np.log1p) 
# Independent variables for each model
models = {
    'Demographics (D1)': ['Or_population_density', 
                          'log_Or_unemployment_rate', 
                          #'log_Or_white_population',
                          #'log_black_population',
                          #'log_Or_asian_population', 
                          #'log_Or_all_others',
                          'Or_median_household_income',
                          'Or_education_atleast_one_degree'
                         ],

    'D1 + Distance between OD (D2)': ['Or_population_density', 
                                      'log_Or_unemployment_rate',
                                      #'log_Or_white_population',
                                      #'log_black_population',
                                      #'log_Or_asian_population',
                                      #'log_Or_all_others',
                                      'Or_median_household_income',
                                      'Or_education_atleast_one_degree', 
                                      'log_distance'],

    'D1 + D2 + Distance from fire (D3)': ['Or_population_density', 
                                          'log_Or_unemployment_rate', 
                                          #'log_Or_white_population',
                                          #'log_black_population',
                                          #'log_Or_asian_population',
                                          #'log_Or_all_others',
                                          'Or_median_household_income',
                                          'Or_education_atleast_one_degree', 
                                          'dist_epicentre',
                                          'log_distance'],

    'D1 + D2 + D3 + Pre-disaster mobility': ['Or_population_density', 
                                             'log_Or_unemployment_rate', 
                                             #'log_Or_white_population',
                                             #'log_black_population',
                                             #'log_Or_asian_population', 
                                             #'log_Or_all_others',
                                             'Or_median_household_income',
                                             'Or_education_atleast_one_degree', 
                                             'dist_epicentre',
                                             'log_distance',
                                             'avg_entropy',
                                            'radius_of_gyration'
                                            ]
}

In [ ]:
# Define the color palette
palette = ['#76FF7B', '#15B018', '#008000', '#054907']

# Initialize dictionaries to store R-squared values for both SCI and Homophily models
r_squared_values_sci = {}
r_squared_values_homo = {}

# Loop over the models to calculate R-squared values
for model_name, independent_vars in models.items():
    # Standardize the independent variables
    X = df[independent_vars].copy()
    scaler = StandardScaler()
    X = scaler.fit_transform(X)

    # Add a constant term
    X_with_const = sm.add_constant(X)

    # Calculate R-squared for SCI
    model_SCI = sm.OLS(Y_SCI, X_with_const).fit()
    r_squared_values_sci[model_name] = model_SCI.rsquared

    # Calculate R-squared for Homophily
    model_homophily = sm.OLS(Y_homo, X_with_const).fit()
    r_squared_values_homo[model_name] = model_homophily.rsquared

# Convert R-squared values to DataFrames for bar plots
r_squared_df_sci = pd.DataFrame(list(r_squared_values_sci.items()), columns=['Model', 'R-squared'])
r_squared_df_homo = pd.DataFrame(list(r_squared_values_homo.items()), columns=['Model', 'R-squared'])

# Plot horizontal R-squared bar plots for SCI using the custom palette
plt.figure(figsize=(5, 3))
sns.barplot(y='Model', x='R-squared', data=r_squared_df_sci, palette=palette)
plt.title("Familiarity")
plt.xlabel("R-squared")
plt.ylabel("Model")
plt.tight_layout()
plt.show()

# Plot horizontal R-squared bar plots for Homophily using the custom palette
plt.figure(figsize=(5, 3))
sns.barplot(y='Model', x='R-squared', data=r_squared_df_homo, palette=palette)
plt.title("Similarity")
plt.xlabel("R-squared")
plt.ylabel("Model")
plt.tight_layout()
plt.show()

In [ ]:
# Define the color palette
palette = ['#76FF7B', '#15B018', '#008000', '#054907']

# Define the order of variables for plotting
variable_labels = {
    'Or_population_density': 'Population\nDensity',
    'log_Or_unemployment_rate': 'Unemployment Rate',
    #'log_Or_white_population': 'White\nPopulation',
    #'log_black_population': 'Black\nPopulation',
    #'log_Or_all_others' : 'All_other_races',
    #'log_Or_asian_population': 'Asian\nPopulation',
    'Or_median_household_income': 'Median\nIncome',
    'Or_education_atleast_one_degree': 'Education',
    'log_distance': 'Distance origin\nto destination',
    'dist_epicentre': 'Distance\nfrom Fire',
    'avg_entropy': 'Entropy',
    'radius_of_gyration': 'Radius of\nGyration'
}

ordered_vars = list(variable_labels.keys())

# Assign colors to the variables based on their model groupings
color_mapping = {}

# Assign the first color to variables in the first model
for var in models['Demographics (D1)']:
    color_mapping[var] = palette[0]

# Assign the second color to variables added in the second model
for var in models['D1 + Distance between OD (D2)']:
    if var not in color_mapping:  # Avoid overwriting
        color_mapping[var] = palette[1]

# Assign the third color to variables added in the third model
for var in models['D1 + D2 + Distance from fire (D3)']:
    if var not in color_mapping:
        color_mapping[var] = palette[2]

# Assign the fourth color to variables added in the fourth model
for var in models['D1 + D2 + D3 + Pre-disaster mobility']:
    if var not in color_mapping:
        color_mapping[var] = palette[3]

def plot_significant_coefficients(df, Y, model_name, color_mapping, variable_labels, ordered_vars, significance_level=0.05):
    # Step 1: Get the independent variables for the selected model
    independent_vars = models[model_name]
    
    # Step 2: Standardize the independent variables
    X = df[independent_vars].copy()
    scaler = StandardScaler()
    X = scaler.fit_transform(X)

    # Step 3: Add a constant term for the intercept in the model
    X_with_const = sm.add_constant(X)

    # Step 4: Fit the OLS model using statsmodels
    model = sm.OLS(Y, X_with_const).fit()

    # Step 5: Create `coef_df` with coefficients, standard errors, and p-values
    coef_df = pd.DataFrame({
        'Variable': independent_vars,
        'Coefficient': model.params[1:],  # Exclude intercept
        'Standard_Error': model.bse[1:],  # Exclude intercept
        'P_value': model.pvalues[1:]  # Exclude intercept
    }).round(2)  # Round all values to 2 decimal points

    # Step 6: Filter for statistically significant coefficients only
    significant_coef_df = coef_df[coef_df['P_value'] < significance_level].copy()

    # Step 7: Ensure the variables in `significant_coef_df` are ordered as specified
    significant_coef_df['Variable'] = pd.Categorical(
        significant_coef_df['Variable'], categories=ordered_vars, ordered=True
    )
    significant_coef_df = significant_coef_df.sort_values('Variable')

    # Step 8: Plot significant coefficients with error bars
    fig, ax = plt.subplots(figsize=(4, 5))
    ax.invert_yaxis()

    # Plotting loop: Only plot variables that have mappings in `variable_labels` and `color_mapping`
    for i, row in significant_coef_df.iterrows():
        var_name = row['Variable']
        if var_name in variable_labels and var_name in color_mapping:
            ax.errorbar(row['Coefficient'], variable_labels[var_name], 
                        xerr=row['Standard_Error'], fmt='v', color=color_mapping[var_name],
                        ecolor='black', elinewidth=0.8, capsize=4, markersize=8, linestyle='None',
                        markerfacecolor=color_mapping[var_name], markeredgewidth=1.5, alpha=0.5)
        else:
            print(f"Warning: Missing mapping for {var_name}")

    # Add vertical line at zero and labels
    ax.axvline(0, color='black', linewidth=0.8, linestyle='--')
    ax.set_xlabel('Beta Coefficients')
    ax.set_ylabel('Variables')
    ax.set_title(f'Significant Coefficients for {model_name}')
    plt.tight_layout()
    plt.show()

    # Print coefficients, standard errors, and p-values
    print("\nSignificant Coefficients Table:")
    print(significant_coef_df)

# Plot only significant coefficients for the "D1 + D2 + D3 + Pre-disaster mobility" model
plot_significant_coefficients(df, Y_SCI, 'D1 + D2 + D3 + Pre-disaster mobility', color_mapping, variable_labels, ordered_vars)
plot_significant_coefficients(df, Y_homo, 'D1 + D2 + D3 + Pre-disaster mobility', color_mapping, variable_labels, ordered_vars)


In [ ]:
# Calculate the additional contribution to R-squared for each model for log_D1_scaled_sci
contribution_sci = {}
prev_r2 = 0
for model_name, r2 in r_squared_values_sci.items():
    contribution_sci[model_name] = r2 - prev_r2
    prev_r2 = r2

# Calculate the additional contribution to R-squared for each model for log_homophily
contribution_homo = {}
prev_r2 = 0
for model_name, r2 in r_squared_values_homo.items():
    contribution_homo[model_name] = r2 - prev_r2
    prev_r2 = r2

# Convert the contributions to DataFrames for plotting
contribution_df_sci = pd.DataFrame(list(contribution_sci.items()), columns=['Model', 'Contribution'])
contribution_df_homo = pd.DataFrame(list(contribution_homo.items()), columns=['Model', 'Contribution'])

# Save the first plot (Familiarity) in 300 DPI
plt.figure(figsize=(5, 3))
sns.barplot(y='Model', x='Contribution', data=contribution_df_sci, palette=palette)
plt.title("Additional Contribution to R-squared (Familiarity)")
plt.xlabel("Additional R-squared")
plt.ylabel("Model")
plt.tight_layout()
plt.savefig('contribution_sci.png', dpi=300)
plt.show()

# Save the second plot (Similarity) in 300 DPI
plt.figure(figsize=(5, 3))
sns.barplot(y='Model', x='Contribution', data=contribution_df_homo, palette=palette)
plt.title("Additional Contribution to R-squared (Similarity)")
plt.xlabel("Additional R-squared")
plt.ylabel("Model")
plt.tight_layout()
plt.savefig('contribution_homo.png', dpi=300)
plt.show()